In [1]:
import numpy as np
import matplotlib.pyplot as plt
import pickle

from math import log, sqrt
from time import time
from pprint import pprint

from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score as AUC, log_loss, accuracy_score as accuracy
from sklearn.metrics import (mean_squared_error as MSE, mean_absolute_error as MAE, r2_score as R2,
                             explained_variance_score as EVS)
from sklearn.preprocessing import StandardScaler, RobustScaler, MinMaxScaler, MaxAbsScaler

from keras.models import Sequential
from keras.layers.core import Dense, Dropout
from keras.layers.normalization import BatchNormalization as BatchNorm
from keras.callbacks import EarlyStopping, ModelCheckpoint
from keras.layers.advanced_activations import *
from keras.models import load_model

%load_ext autoreload
%autoreload 2

%matplotlib inline

plt.rcParams['figure.figsize'] = (10, 8)

/home/bulent/anaconda3/lib/python3.6/site-packages/h5py/__init__.py:34: FutureWarning: Conversion of the second argument of issubdtype from `float` to `np.floating` is deprecated. In future, it will be treated as `np.float64 == np.dtype(float).type`.
  from ._conv import register_converters as _register_converters
Using TensorFlow backend.


In [2]:
with open('stations-6to31.pkl', 'rb') as f:
    datas = pickle.load(f)
    
x_train6_ = datas['x_train6']
y_train6 = datas['y_train6']
x_train_ = datas['x_train']
y_train = datas['y_train']
x_dev_ = datas['x_dev']
y_dev = datas['y_dev']
x_test_ = datas['x_test']
y_test = datas['y_test']

print(f'x_train shape: {x_train_.shape}, y_train shape: {y_train.shape}')
print(f'x_train6 shape: {x_train6_.shape}, y_train6 shape: {y_train6.shape}')
print(f'x_dev shape: {x_dev_.shape}, y_dev shape: {y_dev.shape}')
print(f'x_test shape: {x_test_.shape}, y_test shape: {y_test.shape}')

x_train shape: (52416, 5), y_train shape: (52416,)
x_train6 shape: (10654, 5), y_train6 shape: (10654,)
x_dev shape: (9011, 5), y_dev shape: (9011,)
x_test shape: (16899, 5), y_test shape: (16899,)


From the best 21 configurations modes of respective categories are as follows.

**Initializer:** normal

**Layers:** 2

**Batch Size:** 64

**Optimizer:** adamax

**Shuffle:** True

**Scaler:** RobustScaler

**Loss:** mean_absolute_error

In [3]:
def scale_data(scaler, datas):
    # scaler is a scaling function from sklearn library
    # datas is a dictionary, containing 3 sets of x_data with keys - x_train, x_dev, x_test
    # fit on x_train and return the transformed sets of data
    
    x_train = scaler.fit_transform(datas['x_train'].astype(float))
    x_dev = scaler.transform(datas['x_dev'].astype(float))
    x_test = scaler.transform(datas['x_test'].astype(float))
    
    transformed = {'x_train': x_train, 'x_dev': x_dev, 'x_test': x_test}
    return transformed

data6_ = {'x_train': x_train6_, 'x_dev': x_dev_, 'x_test': x_test_}
data_ = {'x_train': x_train_, 'x_dev': x_dev_, 'x_test': x_test_}

data6 = scale_data(RobustScaler(), data6_)
data = scale_data(RobustScaler(), data_)

x_train6 = data6['x_train']
x_train = data['x_train']

x_dev6 = data6['x_dev']
x_dev = data['x_dev']

x_test6 = data6['x_test']
x_test = data['x_test']

print(f'x_train shape: {x_train.shape}, y_train shape: {y_train.shape}')
print(f'x_train6 shape: {x_train6.shape}, y_train6 shape: {y_train6.shape}')
print(f'x_dev shape: {x_dev.shape}, y_dev shape: {y_dev.shape}')
print(f'x_test shape: {x_test.shape}, y_test shape: {y_test.shape}')

x_train shape: (52416, 5), y_train shape: (52416,)
x_train6 shape: (10654, 5), y_train6 shape: (10654,)
x_dev shape: (9011, 5), y_dev shape: (9011,)
x_test shape: (16899, 5), y_test shape: (16899,)


In [4]:
def radstimator(h1=20, h2=15, num_vars=5):
    init = 'normal'
    
    model = Sequential()
    model.add( Dense( h1, kernel_initializer=init, input_dim=num_vars ))
    model.add( PReLU( alpha_initializer=init ))
    model.add( BatchNorm())
    model.add( Dense( h2, kernel_initializer=init ))
    model.add( PReLU( alpha_initializer=init ))
    model.add( Dropout( rate=0.4 ))
    
    model.add( Dense( 1, kernel_initializer=init, activation='linear' ))
    
    return model

In [9]:
print(x_train6_[:5]) # 'Latitude', 'BSH', 'Temperature(avg)', 'Daylength', 'H0'

[[40.141       5.9         4.22916667  9.19499685 13.69287119]
 [40.141       1.3         7.6375      9.20626964 13.74607822]
 [40.141       0.          6.2375      9.21860396 13.80432176]
 [40.141       0.          3.32916667  9.23198817 13.86758193]
 [40.141       0.7         5.06956522  9.24640976 13.93583675]]


In [5]:
data6_3v_ = {'x_train': x_train6_[:, [1, 3, 2]], 'x_dev': x_dev_[:, [1, 3, 2]], 'x_test': x_test_[:, [1, 3, 2]]}
data_3v_ = {'x_train': x_train_[:, [1, 3, 2]], 'x_dev': x_dev_[:, [1, 3, 2]], 'x_test': x_test_[:, [1, 3, 2]]}

data6_3v = scale_data(RobustScaler(), data6_3v_)
data_3v = scale_data(RobustScaler(), data_3v_)

x_train6_3v = data6_3v['x_train']
x_train_3v = data_3v['x_train']

x_dev6_3v = data6_3v['x_dev']
x_dev_3v = data_3v['x_dev']

x_test6_3v = data6_3v['x_test']
x_test_3v = data_3v['x_test']

y_train6_hh0 = y_train6 / x_train6_[:, 4]
y_train_hh0 = y_train / x_train_[:, 4]
y_dev_hh0 = y_dev / x_dev_[:, 4]
y_test_hh0 = y_test / x_test_[:, 4]

print(f'x_train shape: {x_train_3v.shape}, y_train shape: {y_train.shape}')
print(f'x_train6 shape: {x_train6_3v.shape}, y_train6 shape: {y_train6.shape}')
print(f'x_dev shape: {x_dev_3v.shape}, y_dev shape: {y_dev.shape}')
print(f'x_test shape: {x_test_3v.shape}, y_test shape: {y_test.shape}')

x_train shape: (52416, 3), y_train shape: (52416,)
x_train6 shape: (10654, 3), y_train6 shape: (10654,)
x_dev shape: (9011, 3), y_dev shape: (9011,)
x_test shape: (16899, 3), y_test shape: (16899,)


In [6]:
# Train with 6 stations data, and then with 31. Compare them.
# First for the variables n,N for H/H0.

validation_data6 = ( x_dev6_3v, y_dev_hh0 )
metrics_test6 = []
metrics_train6 = []
metrics_dev6 = []
for i in range(50):
    rads = radstimator(20, 15, 3)
    rads.compile(optimizer='adamax', loss='mean_absolute_error')

    early_stopping = EarlyStopping( monitor = 'val_loss', patience = 10, verbose = 0 )
    filepath = './6to31stats-3vars-hh0/6stations-hh0-nNT {}.h5'.format(i+1)
    checkpointer = ModelCheckpoint(filepath, monitor='val_loss', verbose=0, save_best_only=True )
    history = rads.fit( x_train6_3v, y_train6_hh0, epochs = 250, batch_size = 64, shuffle = True, 
                         validation_data = validation_data6, callbacks = [ early_stopping, checkpointer ], verbose=0)

    p = rads.predict( x_train6_3v, batch_size = 64 )

    mse = MSE( y_train6_hh0, p )
    rmse = sqrt( mse )
    mae = MAE( y_train6_hh0, p )
    r2 = R2( y_train6_hh0, p )
    evs = EVS( y_train6_hh0, p )
    
    print('C{:02d} » RMSE: {:.4f}, MAE: {:.4f}, R2: {:.4f}, EVS: {:.4f}, '.format(i+1,rmse, mae, r2, evs))

C01 » RMSE: 0.1332, MAE: 0.0728, R2: 0.6446, EVS: 0.6484, 
C02 » RMSE: 0.1350, MAE: 0.0741, R2: 0.6347, EVS: 0.6477, 
C03 » RMSE: 0.1325, MAE: 0.0729, R2: 0.6480, EVS: 0.6487, 
C04 » RMSE: 0.1334, MAE: 0.0729, R2: 0.6435, EVS: 0.6482, 
C05 » RMSE: 0.1327, MAE: 0.0746, R2: 0.6472, EVS: 0.6497, 
C06 » RMSE: 0.1358, MAE: 0.0744, R2: 0.6302, EVS: 0.6482, 
C07 » RMSE: 0.1336, MAE: 0.0723, R2: 0.6424, EVS: 0.6473, 
C08 » RMSE: 0.1362, MAE: 0.0723, R2: 0.6283, EVS: 0.6372, 
C09 » RMSE: 0.1343, MAE: 0.0719, R2: 0.6388, EVS: 0.6473, 
C10 » RMSE: 0.1333, MAE: 0.0745, R2: 0.6439, EVS: 0.6485, 
C11 » RMSE: 0.1331, MAE: 0.0732, R2: 0.6449, EVS: 0.6469, 
C12 » RMSE: 0.1337, MAE: 0.0732, R2: 0.6420, EVS: 0.6496, 
C13 » RMSE: 0.1335, MAE: 0.0738, R2: 0.6430, EVS: 0.6502, 
C14 » RMSE: 0.1329, MAE: 0.0744, R2: 0.6461, EVS: 0.6466, 
C15 » RMSE: 0.1340, MAE: 0.0717, R2: 0.6400, EVS: 0.6441, 
C16 » RMSE: 0.1346, MAE: 0.0726, R2: 0.6368, EVS: 0.6497, 
C17 » RMSE: 0.1349, MAE: 0.0736, R2: 0.6356, EVS: 0.6483

In [7]:
# Train with 6 stations data, and then with 31. Compare them.
# First for the variables n,N for H/H0.

validation_data = ( x_dev_3v, y_dev_hh0 )
metrics_test = []
metrics_train = []
metrics_dev = []
for i in range(50):
    rads = radstimator(20, 15, 3)
    rads.compile(optimizer='adamax', loss='mean_absolute_error')

    early_stopping = EarlyStopping( monitor = 'val_loss', patience = 10, verbose = 0 )
    filepath = './6to31stats-3vars-hh0/31stations-hh0-nNT {}.h5'.format(i+1)
    checkpointer = ModelCheckpoint(filepath, monitor='val_loss', verbose=0, save_best_only=True )
    history = rads.fit( x_train_3v, y_train_hh0, epochs = 250, batch_size = 64, shuffle = True, 
                         validation_data = validation_data6, callbacks = [ early_stopping, checkpointer ], verbose=0)

    p = rads.predict( x_train_3v, batch_size = 64 )

    mse = MSE( y_train_hh0, p )
    rmse = sqrt( mse )
    mae = MAE( y_train_hh0, p )
    r2 = R2( y_train_hh0, p )
    evs = EVS( y_train_hh0, p )
    
    print('C{:02d} » RMSE: {:.4f}, MAE: {:.4f}, R2: {:.4f}, EVS: {:.4f}, '.format(i+1, rmse, mae, r2, evs))

C01 » RMSE: 0.1067, MAE: 0.0607, R2: 0.7202, EVS: 0.7206, 
C02 » RMSE: 0.1064, MAE: 0.0601, R2: 0.7217, EVS: 0.7219, 
C03 » RMSE: 0.1072, MAE: 0.0594, R2: 0.7177, EVS: 0.7191, 
C04 » RMSE: 0.1059, MAE: 0.0610, R2: 0.7246, EVS: 0.7246, 
C05 » RMSE: 0.1074, MAE: 0.0595, R2: 0.7165, EVS: 0.7168, 
C06 » RMSE: 0.1076, MAE: 0.0604, R2: 0.7156, EVS: 0.7187, 
C07 » RMSE: 0.1080, MAE: 0.0597, R2: 0.7134, EVS: 0.7140, 
C08 » RMSE: 0.1067, MAE: 0.0599, R2: 0.7202, EVS: 0.7212, 
C09 » RMSE: 0.1064, MAE: 0.0602, R2: 0.7218, EVS: 0.7230, 
C10 » RMSE: 0.1072, MAE: 0.0594, R2: 0.7175, EVS: 0.7185, 
C11 » RMSE: 0.1070, MAE: 0.0598, R2: 0.7184, EVS: 0.7202, 
C12 » RMSE: 0.1067, MAE: 0.0622, R2: 0.7201, EVS: 0.7202, 
C13 » RMSE: 0.1066, MAE: 0.0595, R2: 0.7208, EVS: 0.7213, 
C14 » RMSE: 0.1068, MAE: 0.0595, R2: 0.7196, EVS: 0.7209, 
C15 » RMSE: 0.1073, MAE: 0.0600, R2: 0.7172, EVS: 0.7202, 
C16 » RMSE: 0.1069, MAE: 0.0621, R2: 0.7189, EVS: 0.7204, 
C17 » RMSE: 0.1071, MAE: 0.0595, R2: 0.7183, EVS: 0.7183